# 🐉 HOMEWORK 8: GAME OF THRONES API & POSTGRESQL
## Парсинг данных из API Ice and Fire и сохранение в базу данных PostgreSQL

**Автор:**

**Дата:**

**Описание:** Получение информации о книгах и домах Вестероса через API и сохранение в PostgreSQL

## 1. ИМПОРТ БИБЛИОТЕК

In [ ]:
# Импортируем необходимые библиотеки
import requests
import pandas as pd
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine
import json
import time
import os
from typing import List, Dict

print("✅ Библиотеки успешно импортированы")
print(f"📚 Версии библиотек:")
print(f"   requests: {requests.__version__}")
print(f"   pandas: {pd.__version__}")
print(f"   psycopg2: {psycopg2.__version__}")

## 2. КЛАСС ДЛЯ РАБОТЫ С API ICE AND FIRE

In [ ]:
class GoTAPIParser:
    """Класс для работы с API Ice and Fire"""
    
    BASE_URL = "https://www.anapioficeandfire.com/api"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        print("✅ Инициализирован парсер API Ice and Fire")
    
    def get_all_books(self) -> pd.DataFrame:
        """
        Получить информацию обо всех книгах
        
        Returns:
        pd.DataFrame: DataFrame с информацией о книгах
        """
        print("\n📚 Получение информации о книгах...")
        url = f"{self.BASE_URL}/books"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            books_data = response.json()
            
            # Создаем DataFrame
            df_books = pd.DataFrame(books_data)
            
            # Оставляем только нужные колонки
            columns_to_keep = [
                'url', 'name', 'isbn', 'authors', 'numberOfPages', 
                'publisher', 'country', 'mediaType', 'released'
            ]
            
            # Оставляем только существующие колонки
            existing_columns = [col for col in columns_to_keep if col in df_books.columns]
            df_books = df_books[existing_columns]
            
            print(f"✅ Получено {len(df_books)} книг")
            return df_books
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Ошибка при получении книг: {e}")
            return pd.DataFrame()
    
    def get_all_houses(self, limit: int = None) -> pd.DataFrame:
        """
        Получить информацию обо всех домах Вестероса
        
        Args:
        limit: максимальное количество домов (если None - все)
        
        Returns:
        pd.DataFrame: DataFrame с информацией о домах
        """
        print("\n🏰 Получение информации о всех домах...")
        url = f"{self.BASE_URL}/houses"
        all_houses = []
        page = 1
        
        try:
            while True:
                params = {'page': page, 'pageSize': 50}
                response = self.session.get(url, params=params)
                response.raise_for_status()
                
                houses_data = response.json()
                if not houses_data:
                    break
                
                all_houses.extend(houses_data)
                print(f"   Страница {page}: +{len(houses_data)} домов")
                
                # Проверяем лимит
                if limit and len(all_houses) >= limit:
                    all_houses = all_houses[:limit]
                    break
                    
                # Проверяем, есть ли следующая страница
                if len(houses_data) < 50:
                    break
                    
                page += 1
                time.sleep(0.5)  # Задержка чтобы не перегружать API
            
            # Создаем DataFrame
            df_houses = pd.DataFrame(all_houses)
            
            # Оставляем только нужные колонки
            columns_to_keep = [
                'url', 'name', 'region', 'coatOfArms', 'words', 
                'titles', 'seats', 'currentLord', 'heir', 'overlord',
                'founded', 'founder', 'diedOut', 'ancestralWeapons',
                'cadetBranches', 'swornMembers'
            ]
            
            # Оставляем только существующие колонки
            existing_columns = [col for col in columns_to_keep if col in df_houses.columns]
            df_houses = df_houses[existing_columns]
            
            print(f"✅ Всего получено {len(df_houses)} домов")
            return df_houses
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Ошибка при получении домов: {e}")
            return pd.DataFrame()
    
    def get_houses_with_motto(self, limit: int = None) -> pd.DataFrame:
        """
        Получить информацию о домах Вестероса, у которых есть девиз
        
        Args:
        limit: максимальное количество домов (если None - все)
        
        Returns:
        pd.DataFrame: DataFrame с информацией о домах с девизом
        """
        print("\n📜 Получение информации о домах с девизом...")
        url = f"{self.BASE_URL}/houses"
        all_houses = []
        page = 1
        
        try:
            while True:
                # Используем параметр hasWords для фильтрации домов с девизом
                params = {
                    'page': page, 
                    'pageSize': 50,
                    'hasWords': 'true'  # Фильтр: дома с девизом
                }
                
                response = self.session.get(url, params=params)
                response.raise_for_status()
                
                houses_data = response.json()
                if not houses_data:
                    break
                
                all_houses.extend(houses_data)
                print(f"   Страница {page}: +{len(houses_data)} домов")
                
                # Проверяем лимит
                if limit and len(all_houses) >= limit:
                    all_houses = all_houses[:limit]
                    break
                    
                # Проверяем, есть ли следующая страница
                if len(houses_data) < 50:
                    break
                    
                page += 1
                time.sleep(0.5)  # Задержка чтобы не перегружать API
            
            # Создаем DataFrame
            df_houses_with_motto = pd.DataFrame(all_houses)
            
            # Оставляем только нужные колонки
            columns_to_keep = [
                'url', 'name', 'region', 'coatOfArms', 'words', 
                'titles', 'seats', 'currentLord', 'heir', 'overlord',
                'founded', 'founder', 'diedOut', 'ancestralWeapons'
            ]
            
            # Оставляем только существующие колонки
            existing_columns = [col for col in columns_to_keep if col in df_houses_with_motto.columns]
            df_houses_with_motto = df_houses_with_motto[existing_columns]
            
            # Удаляем дома с пустым девизом (на всякий случай)
            df_houses_with_motto = df_houses_with_motto[
                df_houses_with_motto['words'].notna() & 
                (df_houses_with_motto['words'] != '')
            ]
            
            print(f"✅ Всего получено {len(df_houses_with_motto)} домов с девизом")
            return df_houses_with_motto
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Ошибка при получении домов с девизом: {e}")
            return pd.DataFrame()

## 3. КЛАСС ДЛЯ РАБОТЫ С POSTGRESQL

In [ ]:
class PostgreSQLManager:
    """Класс для работы с PostgreSQL"""
    
    def __init__(self, host='localhost', port=5432, database='got_db', 
                 user='postgres', password='password'):
        """
        Инициализация подключения к PostgreSQL
        
        Args:
        host: хост базы данных
        port: порт базы данных
        database: имя базы данных
        user: имя пользователя
        password: пароль пользователя
        """
        self.host = host
        self.port = port
        self.database = database
        self.user = user
        self.password = password
        
        # Строка подключения для SQLAlchemy
        self.engine = None
        self.connection = None
        
        print(f"✅ Инициализирован менеджер PostgreSQL для БД '{database}'")
    
    def connect(self):
        """Установка соединения с базой данных"""
        try:
            # Создаем engine для SQLAlchemy
            self.engine = create_engine(
                f'postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}'
            )
            
            # Подключение для psycopg2
            self.connection = psycopg2.connect(
                host=self.host,
                port=self.port,
                database=self.database,
                user=self.user,
                password=self.password
            )
            print(f"✅ Успешное подключение к базе данных {self.database}")
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения к базе данных: {e}")
            return False
    
    def create_database(self):
        """Создание базы данных если не существует"""
        try:
            # Подключаемся к базе данных postgres для создания новой БД
            conn = psycopg2.connect(
                host=self.host,
                port=self.port,
                database='postgres',
                user=self.user,
                password=self.password
            )
            conn.autocommit = True
            cursor = conn.cursor()
            
            # Проверяем существует ли база данных
            cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (self.database,))
            exists = cursor.fetchone()
            
            if not exists:
                cursor.execute(sql.SQL("CREATE DATABASE {}").format(
                    sql.Identifier(self.database)
                ))
                print(f"✅ База данных '{self.database}' создана")
            else:
                print(f"ℹ️ База данных '{self.database}' уже существует")
            
            cursor.close()
            conn.close()
            return True
            
        except Exception as e:
            print(f"❌ Ошибка при создании базы данных: {e}")
            return False
    
    def create_tables(self):
        """Создание таблиц в базе данных"""
        if not self.connection:
            print("❌ Нет подключения к базе данных")
            return False
        
        try:
            cursor = self.connection.cursor()
            
            # Таблица книг
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS books (
                    id SERIAL PRIMARY KEY,
                    url TEXT UNIQUE,
                    name TEXT,
                    isbn TEXT,
                    authors TEXT[],
                    number_of_pages INTEGER,
                    publisher TEXT,
                    country TEXT,
                    media_type TEXT,
                    released DATE,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            
            # Таблица всех домов
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS houses (
                    id SERIAL PRIMARY KEY,
                    url TEXT UNIQUE,
                    name TEXT,
                    region TEXT,
                    coat_of_arms TEXT,
                    words TEXT,
                    titles TEXT[],
                    seats TEXT[],
                    current_lord TEXT,
                    heir TEXT,
                    overlord TEXT,
                    founded TEXT,
                    founder TEXT,
                    died_out TEXT,
                    ancestral_weapons TEXT[],
                    cadet_branches TEXT[],
                    sworn_members TEXT[],
                    has_motto BOOLEAN DEFAULT FALSE,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            
            # Таблица домов с девизом
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS houses_with_motto (
                    id SERIAL PRIMARY KEY,
                    url TEXT UNIQUE,
                    name TEXT,
                    region TEXT,
                    coat_of_arms TEXT,
                    words TEXT,
                    titles TEXT[],
                    seats TEXT[],
                    current_lord TEXT,
                    heir TEXT,
                    overlord TEXT,
                    founded TEXT,
                    founder TEXT,
                    died_out TEXT,
                    ancestral_weapons TEXT[],
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            
            self.connection.commit()
            cursor.close()
            print("✅ Таблицы созданы успешно")
            return True
            
        except Exception as e:
            print(f"❌ Ошибка при создании таблиц: {e}")
            return False
    
    def save_to_postgres(self, df: pd.DataFrame, table_name: str, if_exists: str = 'replace'):
        """
        Сохранение DataFrame в таблицу PostgreSQL
        
        Args:
        df: DataFrame для сохранения
        table_name: имя таблицы в базе данных
        if_exists: 'replace' - заменить, 'append' - добавить
        """
        if df.empty:
            print(f"⚠️ DataFrame для таблицы {table_name} пустой")
            return False
        
        if not self.engine:
            print("❌ Нет подключения к базе данных")
            return False
        
        try:
            # Используем SQLAlchemy для удобного сохранения
            df.to_sql(
                name=table_name,
                con=self.engine,
                if_exists=if_exists,
                index=False,
                method='multi'
            )
            print(f"✅ Данные сохранены в таблицу '{table_name}' ({len(df)} записей)")
            return True
            
        except Exception as e:
            print(f"❌ Ошибка при сохранении в таблицу {table_name}: {e}")
            return False
    
    def get_table_info(self, table_name: str):
        """Получение информации о таблице"""
        try:
            query = f"SELECT COUNT(*) as count FROM {table_name}"
            df = pd.read_sql(query, self.engine)
            count = df['count'].iloc[0]
            print(f"   📊 Таблица '{table_name}': {count} записей")
            return count
        except Exception as e:
            print(f"   ❌ Ошибка при получении информации о таблице {table_name}: {e}")
            return 0
    
    def execute_query(self, query: str):
        """Выполнение SQL запроса и возврат результатов"""
        try:
            df = pd.read_sql(query, self.engine)
            return df
        except Exception as e:
            print(f"❌ Ошибка при выполнении запроса: {e}")
            return None
    
    def close(self):
        """Закрытие соединения с базой данных"""
        if self.connection:
            self.connection.close()
            print("🔒 Соединение с базой данных закрыто")

## 4. ПОЛУЧЕНИЕ ДАННЫХ ИЗ API

In [ ]:
print("=" * 60)
print("🐉 ПОЛУЧЕНИЕ ДАННЫХ ИЗ API ICE AND FIRE")
print("=" * 60)

# Создаем парсер
parser = GoTAPIParser()

# Получаем данные о книгах
df_books = parser.get_all_books()

# Получаем данные о всех домах (ограничим для быстроты)
df_all_houses = parser.get_all_houses(limit=50)

# Получаем данные о домах с девизом
df_houses_with_motto = parser.get_houses_with_motto(limit=50)

## 5. ПРОСМОТР ПОЛУЧЕННЫХ ДАННЫХ

In [ ]:
print("=" * 60)
print("📊 ПРОСМОТР ПОЛУЧЕННЫХ ДАННЫХ")
print("=" * 60)

# Книги
if not df_books.empty:
    print("\n📚 КНИГИ:")
    print("-" * 40)
    print(f"Всего книг: {len(df_books)}")
    print("\nПервые 5 книг:")
    display(df_books[['name', 'authors', 'numberOfPages', 'released']].head())

# Все дома
if not df_all_houses.empty:
    print("\n🏰 ВСЕ ДОМА:")
    print("-" * 40)
    print(f"Всего домов: {len(df_all_houses)}")
    print("\nПервые 5 домов:")
    display(df_all_houses[['name', 'region', 'words']].head())

# Дома с девизом
if not df_houses_with_motto.empty:
    print("\n📜 ДОМА С ДЕВИЗОМ:")
    print("-" * 40)
    print(f"Всего домов с девизом: {len(df_houses_with_motto)}")
    print("\nПервые 5 домов с девизом:")
    display(df_houses_with_motto[['name', 'words']].head())

## 6. НАСТРОЙКА ПОДКЛЮЧЕНИЯ К POSTGRESQL

In [ ]:
# ⚠️ ИЗМЕНИТЕ ЭТИ ПАРАМЕТРЫ ПОД ВАШУ УСТАНОВКУ POSTGRESQL
db_config = {
    'host': 'localhost',      # хост базы данных
    'port': 5432,             # порт (по умолчанию 5432)
    'database': 'got_db',      # имя базы данных
    'user': 'postgres',        # имя пользователя
    'password': 'password'     # 🔑 ИЗМЕНИТЕ НА ВАШ ПАРОЛЬ
}

print("⚙️ Параметры подключения к PostgreSQL:")
print(f"   Хост: {db_config['host']}")
print(f"   Порт: {db_config['port']}")
print(f"   База данных: {db_config['database']}")
print(f"   Пользователь: {db_config['user']}")

## 7. РАБОТА С POSTGRESQL

In [ ]:
print("=" * 60)
print("🗄️ РАБОТА С POSTGRESQL")
print("=" * 60)

# Создаем менеджер базы данных
db_manager = PostgreSQLManager(**db_config)

# Создаем базу данных если не существует
db_manager.create_database()

# Подключаемся к базе данных
if db_manager.connect():
    # Создаем таблицы
    db_manager.create_tables()
    
    # Сохраняем данные в PostgreSQL
    print("\n💾 Сохранение данных в PostgreSQL:")
    
    # Книги
    if not df_books.empty:
        # Переименовываем колонки для PostgreSQL
        df_books_db = df_books.rename(columns={
            'numberOfPages': 'number_of_pages',
            'mediaType': 'media_type'
        })
        db_manager.save_to_postgres(df_books_db, 'books')
    
    # Все дома
    if not df_all_houses.empty:
        # Переименовываем колонки и добавляем флаг девиза
        df_houses_db = df_all_houses.rename(columns={
            'coatOfArms': 'coat_of_arms',
            'currentLord': 'current_lord',
            'diedOut': 'died_out',
            'ancestralWeapons': 'ancestral_weapons',
            'cadetBranches': 'cadet_branches',
            'swornMembers': 'sworn_members'
        })
        # Добавляем флаг наличия девиза
        df_houses_db['has_motto'] = df_houses_db['words'].notna() & (df_houses_db['words'] != '')
        db_manager.save_to_postgres(df_houses_db, 'houses')
    
    # Дома с девизом
    if not df_houses_with_motto.empty:
        df_houses_motto_db = df_houses_with_motto.rename(columns={
            'coatOfArms': 'coat_of_arms',
            'currentLord': 'current_lord',
            'diedOut': 'died_out',
            'ancestralWeapons': 'ancestral_weapons'
        })
        db_manager.save_to_postgres(df_houses_motto_db, 'houses_with_motto')

## 8. СТАТИСТИКА БАЗЫ ДАННЫХ

In [ ]:
print("\n📊 СТАТИСТИКА БАЗЫ ДАННЫХ:")
print("-" * 40)
db_manager.get_table_info('books')
db_manager.get_table_info('houses')
db_manager.get_table_info('houses_with_motto')

## 9. ПРИМЕРЫ SQL ЗАПРОСОВ

In [ ]:
print("\n📝 ПРИМЕРЫ SQL ЗАПРОСОВ:")
print("=" * 60)

# Запрос 1: Топ-5 домов с самым длинным девизом
print("\n1. Топ-5 домов с самым длинным девизом:")
query1 = """
    SELECT name, words, LENGTH(words) as motto_length
    FROM houses_with_motto
    WHERE words IS NOT NULL AND words != ''
    ORDER BY motto_length DESC
    LIMIT 5
"""
result1 = db_manager.execute_query(query1)
if result1 is not None:
    display(result1)

# Запрос 2: Количество домов по регионам
print("\n2. Количество домов по регионам:")
query2 = """
    SELECT region, COUNT(*) as house_count
    FROM houses
    WHERE region IS NOT NULL AND region != ''
    GROUP BY region
    ORDER BY house_count DESC
    LIMIT 10
"""
result2 = db_manager.execute_query(query2)
if result2 is not None:
    display(result2)

# Запрос 3: Книги, отсортированные по дате выхода
print("\n3. Книги, отсортированные по дате выхода:")
query3 = """
    SELECT name, authors, released, number_of_pages
    FROM books
    WHERE released IS NOT NULL
    ORDER BY released DESC
    LIMIT 5
"""
result3 = db_manager.execute_query(query3)
if result3 is not None:
    display(result3)

# Запрос 4: Дома с самым большим количеством титулов
print("\n4. Дома с самым большим количеством титулов:")
query4 = """
    SELECT name, array_length(titles, 1) as titles_count
    FROM houses
    WHERE titles IS NOT NULL
    ORDER BY titles_count DESC
    LIMIT 5
"""
result4 = db_manager.execute_query(query4)
if result4 is not None:
    display(result4)

## 10. ЭКСПОРТ ДАННЫХ В CSV

In [ ]:
print("\n💾 ЭКСПОРТ ДАННЫХ В CSV ФАЙЛЫ:")
print("=" * 60)

# Создаем папку для экспорта если её нет
export_dir = 'got_export'
if not os.path.exists(export_dir):
    os.makedirs(export_dir)
    print(f"📁 Создана папка: {export_dir}")

# Экспорт книг
if not df_books.empty:
    books_file = os.path.join(export_dir, 'got_books.csv')
    df_books.to_csv(books_file, index=False, encoding='utf-8-sig')
    print(f"✅ Книги сохранены: {books_file}")

# Экспорт всех домов
if not df_all_houses.empty:
    houses_file = os.path.join(export_dir, 'got_all_houses.csv')
    df_all_houses.to_csv(houses_file, index=False, encoding='utf-8-sig')
    print(f"✅ Все дома сохранены: {houses_file}")

# Экспорт домов с девизом
if not df_houses_with_motto.empty:
    motto_file = os.path.join(export_dir, 'got_houses_with_motto.csv')
    df_houses_with_motto.to_csv(motto_file, index=False, encoding='utf-8-sig')
    print(f"✅ Дома с девизом сохранены: {motto_file}")

## 11. ВЫВОДЫ

### 📌 ИТОГИ РАБОТЫ:

**1. ПОЛУЧЕННЫЕ ДАННЫЕ ИЗ API:**
   - Книги: всего получено информации о всех книгах серии
   - Дома Вестероса: собрана информация о домах, включая девизы, регионы, титулы
   - Дома с девизом: отфильтрованы дома, имеющие девиз

**2. РАБОТА С POSTGRESQL:**
   - Создана база данных 'got_db'
   - Созданы таблицы: books, houses, houses_with_motto
   - Данные успешно сохранены в соответствующие таблицы
   - Выполнены SQL запросы для анализа данных

**3. СТАТИСТИКА:**


In [ ]:
# Выводим статистику
print("📊 ФИНАЛЬНАЯ СТАТИСТИКА:")
print("-" * 40)
print(f"📚 Книг: {len(df_books) if not df_books.empty else 0}")
print(f"🏰 Всего домов: {len(df_all_houses) if not df_all_houses.empty else 0}")
print(f"📜 Домов с девизом: {len(df_houses_with_motto) if not df_houses_with_motto.empty else 0}")

if not df_all_houses.empty and not df_houses_with_motto.empty:
    percent = (len(df_houses_with_motto) / len(df_all_houses)) * 100
    print(f"📊 Процент домов с девизом: {percent:.1f}%")

**4. ДОПОЛНИТЕЛЬНО:**
   - Данные экспортированы в CSV файлы для резервного копирования
   - Все запросы к API выполнялись с задержками для соблюдения ограничений
   - Реализована обработка ошибок при подключении к API и БД

**5. ВОЗМОЖНЫЕ УЛУЧШЕНИЯ:**
   - Добавить парсинг персонажей
   - Реализовать связи между таблицами (foreign keys)
   - Добавить индексы для ускорения запросов
   - Создать представления для часто используемых запросов

## 12. ЗАКРЫТИЕ СОЕДИНЕНИЯ

In [ ]:
# Закрываем соединение с базой данных
if 'db_manager' in locals():
    db_manager.close()
    print("✅ Соединение с БД закрыто")

print("\n" + "=" * 60)
print("✅ ВЫПОЛНЕНИЕ ЗАДАНИЯ ЗАВЕРШЕНО")
print("=" * 60)